# DFK Model → GGUF Converter
Konversi model `aitf-komdigi/KomdigiITS-8B-DFK-TextClassification` ke format GGUF Q4_K_M untuk inference CPU gratis di HF Space.

**Estimasi waktu:** ~30-45 menit  |  **Disk dibutuhkan:** ~60 GB

Jalankan cell satu per satu dari atas ke bawah.

In [ ]:
# Cell 1: Install semua dependency
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub', 'transformers', 'sentencepiece', 'gguf', 'numpy'
], check=True)
print('✓ Dependencies installed')

In [ ]:
# Cell 2: Clone llama.cpp
import os, sys, subprocess

LLAMA_DIR  = '/content/llama.cpp'
MODEL_DIR  = '/content/dfk_model'
OUTPUT_F16 = '/content/model-f16.gguf'
OUTPUT_Q4  = '/content/model-q4_k_m.gguf'

if not os.path.exists(LLAMA_DIR):
    print('Cloning llama.cpp ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/ggerganov/llama.cpp', LLAMA_DIR], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)
    print('✓ llama.cpp ready')
else:
    print('✓ llama.cpp already exists')

In [ ]:
# Cell 3: Download model (~35 GB)
from huggingface_hub import snapshot_download

HF_TOKEN = ''  # isi token HF kamu jika perlu
MODEL_ID = 'aitf-komdigi/KomdigiITS-8B-DFK-TextClassification'

if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print(f'Downloading {MODEL_ID} (~35 GB) ...')
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
        token=HF_TOKEN or None,
    )
    print('✓ Download selesai')
else:
    print(f'✓ Model sudah ada di {MODEL_DIR}')

print('Files:', sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Cell 4: Patch config.json dan tokenizer_config.json
import json

# --- config.json ---
config_path = os.path.join(MODEL_DIR, 'config.json')
with open(config_path) as f:
    cfg = json.load(f)

changed = False
if cfg.get('model_type') in ('mistral3', 'ministral3'):
    cfg['model_type']    = 'mistral'
    cfg['architectures'] = ['MistralForCausalLM']
    changed = True
    print('Patched: model_type -> mistral')
if 'generation_config' in cfg:
    del cfg['generation_config']
    changed = True
    print('Patched: removed nested generation_config')
if changed:
    with open(config_path, 'w') as f:
        json.dump(cfg, f, indent=2)

# --- tokenizer_config.json ---
tok_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
with open(tok_path) as f:
    tok = json.load(f)
if tok.get('tokenizer_class') == 'TokenizersBackend':
    tok['tokenizer_class'] = 'PreTrainedTokenizerFast'
    with open(tok_path, 'w') as f:
        json.dump(tok, f, indent=2)
    print('Patched: tokenizer_class -> PreTrainedTokenizerFast')

print(f"\nmodel_type    : {cfg['model_type']}")
print(f"tokenizer_class: {tok['tokenizer_class']}")
print('✓ Patches done')

In [ ]:
# Cell 5: Convert ke GGUF f16 (tampilkan error lengkap jika gagal)
convert_script = os.path.join(LLAMA_DIR, 'convert_hf_to_gguf.py')
print(f'Script: {convert_script}')
print(f'Model : {MODEL_DIR}')
print(f'Output: {OUTPUT_F16}')
print('Converting ... (~10-15 menit)\n')

result = subprocess.run(
    [sys.executable, convert_script,
     MODEL_DIR,
     '--outfile', OUTPUT_F16,
     '--outtype', 'f16'],
    text=True, capture_output=True
)

if result.stdout: print('STDOUT:\n', result.stdout[-4000:])
if result.stderr: print('STDERR:\n', result.stderr[-4000:])

if result.returncode != 0:
    print(f'\n❌ GAGAL (exit {result.returncode})')
    print('Copy paste STDERR di atas untuk debugging lebih lanjut.')
else:
    size = os.path.getsize(OUTPUT_F16) / 1e9
    print(f'\n✓ f16 GGUF selesai: {size:.1f} GB')

In [ ]:
# Cell 6: Build llama-quantize dan quantize ke Q4_K_M
quantize_bin = os.path.join(LLAMA_DIR, 'build', 'bin', 'llama-quantize')

if not os.path.exists(quantize_bin):
    print('Building llama-quantize ...')
    build_dir = os.path.join(LLAMA_DIR, 'build')
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(['cmake', '-B', build_dir, LLAMA_DIR,
                    '-DLLAMA_NATIVE=OFF', '-DCMAKE_BUILD_TYPE=Release'],
                   check=True)
    subprocess.run(['cmake', '--build', build_dir,
                    '--target', 'llama-quantize', '-j4'], check=True)
    print('✓ llama-quantize built')

print('Quantizing f16 → Q4_K_M ...')
subprocess.run([quantize_bin, OUTPUT_F16, OUTPUT_Q4, 'Q4_K_M'], check=True)

size = os.path.getsize(OUTPUT_Q4) / 1e9
print(f'\n✓ Q4_K_M GGUF: {size:.1f} GB')

os.remove(OUTPUT_F16)
print('✓ f16 dihapus (hemat disk)')

In [ ]:
# Cell 7: Upload ke HuggingFace Hub
from huggingface_hub import HfApi

HF_TOKEN      = ''  # WAJIB isi token HF kamu
GGUF_REPO     = 'ggapar/KomdigiITS-8B-DFK-GGUF'
GGUF_FILENAME = 'model-q4_k_m.gguf'

assert HF_TOKEN, 'Isi HF_TOKEN dulu!'
assert os.path.exists(OUTPUT_Q4), 'Jalankan cell 5-6 dulu'

api = HfApi(token=HF_TOKEN)
api.create_repo(GGUF_REPO, repo_type='model', exist_ok=True)
print(f'Repo: https://huggingface.co/{GGUF_REPO}')

print(f'Uploading {os.path.getsize(OUTPUT_Q4)/1e9:.1f} GB ...')
api.upload_file(
    path_or_fileobj=OUTPUT_Q4,
    path_in_repo=GGUF_FILENAME,
    repo_id=GGUF_REPO,
    repo_type='model',
    commit_message='Upload DFK GGUF Q4_K_M',
)
print(f'\n✓ Upload selesai!')
print(f'URL: https://huggingface.co/{GGUF_REPO}')
print(f'\nTambahkan ke HF Space secrets:')
print(f'  GGUF_REPO     = {GGUF_REPO}')
print(f'  GGUF_FILENAME = {GGUF_FILENAME}')